In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image, ImageFile, UnidentifiedImageError
import base64
from io import BytesIO
import sys, json, warnings

# --- Suppress warnings for clean JSON output ---
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 1. Load the model
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(
    r'C:\Projects\python\deep_learning\chest_xray_phneumonia_detection_CNN\pneumonia_classifier.pth',
    map_location=torch.device('cpu')
))
model.eval()

# 2. Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. Class labels
class_names = ['Normal', 'Tuberculosis']


In [ ]:
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Load image
image_path = "C:\\Users\\jorda\Downloads\\tb3.jpg"
img = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

input_tensor = transform(img).unsqueeze(0).to(torch.device)

# Choose model layer to inspect (last conv layer)
target_layer = model.layer4[-1]   # For ResNet18

cam = GradCAM(model=model, target_layers=[target_layer])

# Set target class (0, 1, or 2)
targets = [ClassifierOutputTarget(prediction_class)]  # e.g. prediction_class = 1

grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

# Convert PIL to numpy normalized
rgb_img = np.array(img.resize((224,224))) / 255.0

# Overlay heatmap
heatmap_result = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# Save result
cv2.imwrite("heatmap_output.jpg", cv2.cvtColor(heatmap_result, cv2.COLOR_RGB2BGR))
print("✅ Heatmap saved to heatmap_output.jpg")